# Task 5: Fine-Tuning DistilBERT for POS Tagging & Chunking


## Step 0: Install Required Libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

# Install required packages
!pip install transformers datasets seqeval evaluate accelerate nltk -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00


## Task 1: Dataset Selection


In [2]:
import nltk
from nltk.corpus import conll2000
import pandas as pd
from datasets import Dataset, DatasetDict
from collections import Counter
nltk.download('conll2000', quiet=True)
# Parse CoNLL-2000 into a list of dicts with tokens + tag lists
def parse_conll2000(tagged_sents):
    records = []
    for sent in tagged_sents:
        tokens, pos_tags, chunk_tags = [], [], []
        for word, pos, chunk in sent:
            tokens.append(word)
            pos_tags.append(pos)
            chunk_tags.append(chunk)
        records.append({'tokens': tokens, 'pos_tags': pos_tags, 'chunk_tags': chunk_tags})
    return records

train_sents = conll2000.iob_sents('train.txt')  # 8936 sentences
test_sents  = conll2000.iob_sents('test.txt')   # 2012 sentences

train_records = parse_conll2000(train_sents)
test_records  = parse_conll2000(test_sents)
split_idx = int(len(train_records) * 0.9)
val_records = train_records[split_idx:]
train_records = train_records[:split_idx]

# Build a Hugging Face DatasetDict for compatibility with the rest of the pipeline
raw_dataset = DatasetDict({
    'train':      Dataset.from_list(train_records),
    'validation': Dataset.from_list(val_records),
    'test':       Dataset.from_list(test_records)
})

print('Dataset structure:')
print(raw_dataset)
print('\nSample entry (train[0]):')
print(raw_dataset['train'][0])


Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 8042
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 894
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags'],
        num_rows: 2012
    })
})

Sample entry (train[0]):
{'tokens': ['Confidence', 'in', 'the', 'pound', 'is', 'widely', 'expected', 'to', 'take', 'another', 'sharp', 'dive', 'if', 'trade', 'figures', 'for', 'September', ',', 'due', 'for', 'release', 'tomorrow', ',', 'fail', 'to', 'show', 'a', 'substantial', 'improvement', 'from', 'July', 'and', 'August', "'s", 'near-record', 'deficits', '.'], 'pos_tags': ['NN', 'IN', 'DT', 'NN', 'VBZ', 'RB', 'VBN', 'TO', 'VB', 'DT', 'JJ', 'NN', 'IN', 'NN', 'NNS', 'IN', 'NNP', ',', 'JJ', 'IN', 'NN', 'NN', ',', 'VB', 'TO', 'VB', 'DT', 'JJ', 'NN', 'IN', 'NNP', 'CC', 'NNP', 'POS', 'JJ', 'NNS', '.'], 'chunk_tags': ['B-NP', 'B-PP', 'B

In [3]:
# Build label vocab from ALL splits (train + val + test)
# This prevents KeyError for tags that appear only in test (e.g. 'I-LST')
all_splits = ['train', 'validation', 'test']

pos_label_names = sorted(set(
    tag
    for split in all_splits
    for ex in raw_dataset[split]
    for tag in ex['pos_tags']
))

chunk_label_names = sorted(set(
    tag
    for split in all_splits
    for ex in raw_dataset[split]
    for tag in ex['chunk_tags']
))

print(f'POS Tag labels ({len(pos_label_names)}): {pos_label_names}')
print(f'\nChunk Tag labels ({len(chunk_label_names)}): {chunk_label_names}')


POS Tag labels (44): ['#', '$', "''", '(', ')', ',', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB', '``']

Chunk Tag labels (23): ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-UCP', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-UCP', 'I-VP', 'O']


## Task 2: Data Preprocessing


In [4]:

def make_label2id(label_names):
    return {label: i for i, label in enumerate(label_names)}

pos_label2id_map   = make_label2id(pos_label_names)
chunk_label2id_map = make_label2id(chunk_label_names)

def tokenize_and_align_labels(examples, label_column, label2id_map):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True
    )
    all_labels = []
    for i, str_labels in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_id = None
        label_list = []
        for word_id in word_ids:
            if word_id is None:
                label_list.append(-100)                              # special token
            elif word_id != previous_word_id:
                label_list.append(label2id_map[str_labels[word_id]])  # first subword
            else:
                label_list.append(-100)                              # continuation subword
            previous_word_id = word_id
        all_labels.append(label_list)
    tokenized_inputs['labels'] = all_labels
    return tokenized_inputs

print('Label alignment function defined.')


Label alignment function defined.


In [5]:
from transformers import AutoTokenizer
import logging
logging.set_verbosity_error = lambda *a, **k: None  # suppress transformers logs

MODEL_CHECKPOINT = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f'Tokenizer loaded: {MODEL_CHECKPOINT}')


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: distilbert-base-uncased


In [6]:
# Apply tokenization + label alignment for both tasks
pos_tokenized = raw_dataset.map(
    lambda ex: tokenize_and_align_labels(ex, 'pos_tags', pos_label2id_map),
    batched=True
)

chunk_tokenized = raw_dataset.map(
    lambda ex: tokenize_and_align_labels(ex, 'chunk_tags', chunk_label2id_map),
    batched=True
)

print('Tokenized POS dataset features:', pos_tokenized['train'].features)
print('\nSample — input_ids length:', len(pos_tokenized['train'][0]['input_ids']))
print('Sample — labels:', pos_tokenized['train'][0]['labels'])


Map:   0%|          | 0/8042 [00:00<?, ? examples/s]

Map:   0%|          | 0/894 [00:00<?, ? examples/s]

Map:   0%|          | 0/2012 [00:00<?, ? examples/s]

Map:   0%|          | 0/8042 [00:00<?, ? examples/s]

Map:   0%|          | 0/894 [00:00<?, ? examples/s]

Map:   0%|          | 0/2012 [00:00<?, ? examples/s]

Tokenized POS dataset features: {'tokens': List(Value('string')), 'pos_tags': List(Value('string')), 'chunk_tags': List(Value('string')), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}

Sample — input_ids length: 43
Sample — labels: [-100, 18, 13, 10, 18, 38, 26, 36, 31, 33, 10, 14, 18, 13, 18, 21, 13, 19, 5, 14, 13, 18, 18, 5, 33, 31, 33, 10, 14, 18, 13, 19, 8, 19, 23, -100, 14, -100, -100, 21, -100, 6, -100]


## Task 3: Model Setup


In [7]:
from transformers import AutoModelForTokenClassification

# id2label: integer → string label (used for inference display)
pos_id2label  = {i: l for i, l in enumerate(pos_label_names)}
pos_label2id  = pos_label2id_map

pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(pos_label_names),
    id2label=pos_id2label,
    label2id=pos_label2id
)

print(f'POS Model: {len(pos_label_names)} labels')
print(f'Label mapping sample: {dict(list(pos_id2label.items())[:5])}')


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


POS Model: 44 labels
Label mapping sample: {0: '#', 1: '$', 2: "''", 3: '(', 4: ')'}


In [8]:
chunk_id2label = {i: l for i, l in enumerate(chunk_label_names)}
chunk_label2id = chunk_label2id_map

chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(chunk_label_names),
    id2label=chunk_id2label,
    label2id=chunk_label2id
)

print(f'Chunk Model: {len(chunk_label_names)} labels')
print(f'Label mapping sample: {dict(list(chunk_id2label.items())[:5])}')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Chunk Model: 23 labels
Label mapping sample: {0: 'B-ADJP', 1: 'B-ADVP', 2: 'B-CONJP', 3: 'B-INTJ', 4: 'B-LST'}


## Task 4: Training


In [9]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate
import numpy as np

# Data collator pads batches dynamically to the longest sequence in the batch
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Load seqeval metric — standard metric for sequence labeling tasks
seqeval = evaluate.load("seqeval")

print("Data collator and seqeval metric ready.")

Data collator and seqeval metric ready.


In [10]:
# Returns a compute_metrics function for a given label list
def make_compute_metrics(label_names):
    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        # Argmax over last dim to get predicted label IDs
        predictions = np.argmax(logits, axis=-1)

        # Convert IDs → label strings, skip -100 (special/subword tokens)
        true_labels = [
            [label_names[l] for l in label_row if l != -100]
            for label_row in labels
        ]
        true_preds = [
            [label_names[p] for p, l in zip(pred_row, label_row) if l != -100]
            for pred_row, label_row in zip(predictions, labels)
        ]

        results = seqeval.compute(predictions=true_preds, references=true_labels)
        return {
            "precision": results["overall_precision"],
            "recall":    results["overall_recall"],
            "f1":        results["overall_f1"],
            "accuracy":  results["overall_accuracy"],
        }
    return compute_metrics

print("compute_metrics factory defined.")

compute_metrics factory defined.


In [11]:
def get_training_args(output_dir):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy='epoch',   # renamed from evaluation_strategy in newer transformers
        save_strategy='epoch',
        load_best_model_at_end=True,
        logging_steps=100,
        report_to='none'
    )

print('TrainingArguments factory defined.')


TrainingArguments factory defined.


In [12]:
# =============================================
# EXPERIMENT A: Fine-tune for POS Tagging
# =============================================

pos_trainer = Trainer(
    model=pos_model,
    args=get_training_args('./pos_model'),
    train_dataset=pos_tokenized['train'],
    eval_dataset=pos_tokenized['validation'],
    processing_class=tokenizer,   # 'tokenizer' arg removed in newer transformers
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(pos_label_names)
)

print('Starting POS Tagging training...')
pos_trainer.train()
print('POS Tagging training complete!')


Starting POS Tagging training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.148939,0.120148,0.955887,0.956996,0.956441,0.968539
2,0.100712,0.088368,0.965597,0.965933,0.965765,0.976011
3,0.075075,0.084117,0.967187,0.968197,0.967691,0.977392


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


POS Tagging training complete!


In [13]:
# EXPERIMENT B: Fine-tune for Chunking
chunk_trainer = Trainer(
    model=chunk_model,
    args=get_training_args('./chunk_model'),
    train_dataset=chunk_tokenized['train'],
    eval_dataset=chunk_tokenized['validation'],
    processing_class=tokenizer,   # 'tokenizer' arg removed in newer transformers
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(chunk_label_names)
)

print('Starting Chunking training...')
chunk_trainer.train()
print('Chunking training complete!')


Starting Chunking training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.134255,0.111466,0.955063,0.957810,0.956434,0.972727
2,0.091539,0.088569,0.960004,0.966632,0.963306,0.977772
3,0.079295,0.083848,0.962825,0.968549,0.965679,0.979486


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Chunking training complete!


## Task 5: Evaluation


In [14]:
# Evaluate POS Tagging model on the test set
print("=" * 50)
print("EXPERIMENT A: POS Tagging — Test Set Evaluation")
print("=" * 50)

pos_results = pos_trainer.evaluate(eval_dataset=pos_tokenized["test"])
print(f"  Precision : {pos_results['eval_precision']:.4f}")
print(f"  Recall    : {pos_results['eval_recall']:.4f}")
print(f"  F1 Score  : {pos_results['eval_f1']:.4f}")
print(f"  Accuracy  : {pos_results['eval_accuracy']:.4f}")

EXPERIMENT A: POS Tagging — Test Set Evaluation


  Precision : 0.9629
  Recall    : 0.9652
  F1 Score  : 0.9640
  Accuracy  : 0.9738


In [15]:
# Evaluate Chunking model on the test set
print("=" * 50)
print("EXPERIMENT B: Chunking — Test Set Evaluation")
print("=" * 50)

chunk_results = chunk_trainer.evaluate(eval_dataset=chunk_tokenized["test"])
print(f"  Precision : {chunk_results['eval_precision']:.4f}")
print(f"  Recall    : {chunk_results['eval_recall']:.4f}")
print(f"  F1 Score  : {chunk_results['eval_f1']:.4f}")
print(f"  Accuracy  : {chunk_results['eval_accuracy']:.4f}")

EXPERIMENT B: Chunking — Test Set Evaluation


  Precision : 0.9546
  Recall    : 0.9603
  F1 Score  : 0.9574
  Accuracy  : 0.9735


## Task 6: Inference (10%)


In [16]:
import torch

def predict_tags(sentence, model, tokenizer, id2label):
    words = sentence.split()
    device = next(model.parameters()).device  # auto-detect model device

    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True
    )
    # Move all input tensors to the same device as the model
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=-1)[0].tolist()
    word_ids    = tokenizer(words, is_split_into_words=True).word_ids(batch_index=0)

    result = []
    seen_word_ids = set()
    for token_idx, word_id in enumerate(word_ids):
        if word_id is None or word_id in seen_word_ids:
            continue
        seen_word_ids.add(word_id)
        result.append((words[word_id], id2label[predictions[token_idx]]))

    return result

print('Inference function ready.')


Inference function ready.


In [17]:
# Run inference on sample sentences
test_sentences = [
    "John works at Google in California",
    "The quick brown fox jumps over the lazy dog",
    "She is studying machine learning at the university"
]

for sentence in test_sentences:
    print(f"\nInput: {sentence}")

    pos_tags   = predict_tags(sentence, pos_model,   tokenizer, pos_id2label)
    chunk_tags = predict_tags(sentence, chunk_model, tokenizer, chunk_id2label)

    print(f"{'Word':<20} {'POS Tag':<15} {'Chunk Tag'}")
    print("-" * 50)
    for (word, pos), (_, chunk) in zip(pos_tags, chunk_tags):
        print(f"{word:<20} {pos:<15} {chunk}")


Input: John works at Google in California
Word                 POS Tag         Chunk Tag
--------------------------------------------------
John                 NNP             B-NP
works                VBZ             B-VP
at                   IN              B-PP
Google               NNP             B-NP
in                   IN              B-PP
California           NNP             B-NP

Input: The quick brown fox jumps over the lazy dog
Word                 POS Tag         Chunk Tag
--------------------------------------------------
The                  DT              B-NP
quick                NNP             I-NP
brown                NNP             I-NP
fox                  NNP             I-NP
jumps                VBZ             B-VP
over                 IN              B-PP
the                  DT              B-NP
lazy                 JJ              I-NP
dog                  NN              I-NP

Input: She is studying machine learning at the university
Word                

## Task 7: Comparison — POS Tagging vs Chunking


In [18]:
# Print a side-by-side comparison summary of the two experiments
print("=" * 55)
print("      Experiment Comparison Summary")
print("=" * 55)
print(f"{'Metric':<15} {'POS Tagging':>15} {'Chunking':>15}")
print("-" * 55)
metrics = ["eval_precision", "eval_recall", "eval_f1", "eval_accuracy"]
labels  = ["Precision",      "Recall",      "F1 Score", "Accuracy"]
for metric, label in zip(metrics, labels):
    pos_val   = pos_results.get(metric, 0)
    chunk_val = chunk_results.get(metric, 0)
    print(f"{label:<15} {pos_val:>15.4f} {chunk_val:>15.4f}")
print("=" * 55)
print("\nObservation: POS Tagging generally achieves higher scores")
print("because it operates at word level (simpler).")
print("Chunking is harder — it requires correctly identifying")
print("phrase boundaries using BIO span labels.")

      Experiment Comparison Summary
Metric              POS Tagging        Chunking
-------------------------------------------------------
Precision                0.9629          0.9546
Recall                   0.9652          0.9603
F1 Score                 0.9640          0.9574
Accuracy                 0.9738          0.9735

Observation: POS Tagging generally achieves higher scores
because it operates at word level (simpler).
Chunking is harder — it requires correctly identifying
phrase boundaries using BIO span labels.


## Task 8: Report / Blog

### Fine-Tuning DistilBERT for POS Tagging & Chunking — Observations & Insights

---

#### What is POS Tagging?
Part-of-Speech (POS) Tagging assigns a grammatical label to every word in a sentence — e.g., *Noun (NN)*, *Verb (VBZ)*, *Adjective (JJ)*, *Preposition (IN)*. It is a foundational NLP task used in dependency parsing, information retrieval, and text-to-speech systems.

#### What is Chunking?
Chunking (also called Shallow Parsing or Phrase Detection) groups consecutive words into syntactic phrases using a BIO (Beginning–Inside–Outside) tagging scheme:
- `B-NP`: Beginning of a Noun Phrase
- `I-NP`: Inside a Noun Phrase
- `B-VP`: Beginning of a Verb Phrase
- `O`: Outside any chunk

#### Key Difference
POS tagging labels individual words; chunking labels *spans of words*. Chunking is therefore harder — even one wrong B/I assignment breaks the entire phrase boundary.

---

#### Challenges Faced
1. **Subword Tokenization Mismatch:** BERT's WordPiece tokenizer splits words into subwords (e.g., `"working"` → `["work", "##ing"]`). Since labels exist at the word level, careful alignment was needed — only the first subword receives the word's label; the rest get `-100` to be ignored by the loss.
2. **Special Token Handling:** `[CLS]` and `[SEP]` tokens have no corresponding labels and must be masked with `-100`.
3. **BIO Scheme Evaluation:** seqeval evaluates entire spans, not individual tokens — a misclassified B tag breaks the entire phrase, making chunking evaluation stricter than POS.
4. **Class Imbalance:** The `O` (Outside) tag dominates in chunking, which can bias the model toward over-predicting `O`.

---

#### Observations & Insights
- **DistilBERT** achieved competitive results with only 40% fewer parameters than BERT-base — making it a practical choice for resource-constrained environments.
- POS tagging converges faster and achieves higher F1 scores because each word has an independent label.
- Chunking benefits significantly from BERT's contextual embeddings since phrase boundaries depend on multi-word context.
- The `DataCollatorForTokenClassification` class was essential — it handles dynamic padding while correctly propagating `-100` labels.
- Pre-trained BERT representations capture rich syntactic knowledge, which is why even 3 epochs of fine-tuning yields strong token classification performance.

---

#### Conclusion
Fine-tuning transformer models for token classification is an effective and efficient approach to POS tagging and chunking. The main engineering challenge lies in correctly aligning subword tokens with word-level labels. Once that is handled, Hugging Face's `Trainer` API makes the rest of the pipeline — training, evaluation, and inference — straightforward and reproducible.